# Phase 2 — FinBERT Embedding Baselines

This notebook extends the existing StockNet Phase 1 baseline by introducing **FinBERT-based semantic embeddings** for the text modalities.

The experiment keeps the Phase 1 evaluation protocol and model architectures as unchanged as possible. The main change is the representation of company text:

**Company tweets/text → FinBERT [CLS] embeddings → daily mean embedding → structured features → baseline models**

This notebook is the experimental record for the Phase 2 company-text embedding baseline.


In [2]:
!git clone https://github.com/AdityaMelkote3004/capstone.git
%cd capstone

!find . -maxdepth 3 -type f | sort | head -100

Cloning into 'capstone'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 213 (delta 2), reused 2 (delta 0), pack-reused 205 (from 1)
Receiving objects: 100% (213/213), 15.64 MiB | 27.33 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/capstone
./CLAUDE.md
./config/mmgtfff.yaml
./config/price_baseline.yaml
./data/processed/.gitkeep
./data/README.md
./dataset/stocknet_final_modeling_set.csv
./dataset/stocknet_final_modeling_set.parquet
./experiments/price_only_baseline.ipynb
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sam

In [3]:
%cd /content/capstone

import os

print("Current directory:", os.getcwd())
print("\nTop-level files/folders:")
print(os.listdir())

/content/capstone
Current directory: /content/capstone

Top-level files/folders:
['notes', 'VISUALIZATION_GUIDE.md', '.gitignore', 'data', 'CLAUDE.md', 'requirements.txt', 'scripts', 'quick_start.py', 'results', 'streamlit_app.py', 'STREAMLIT_GUIDE.md', 'dataset', '.git', 'visualize_results.py', 'experiments', 'config', 'README.md', 'preprocessing_notebooks', 'src']


In [4]:
import os

required = [
    "src/data/stocknet_dataset.py",
    "src/models/baselines.py",
    "src/training/trainer.py",
    "src/utils/seed.py",
    "scripts/train_baselines.py",
    "dataset/stocknet_final_modeling_set.parquet",
]

print("Checking Phase 1 files:\n")

for path in required:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {path}")

Checking Phase 1 files:

✓ src/data/stocknet_dataset.py
✓ src/models/baselines.py
✓ src/training/trainer.py
✓ src/utils/seed.py
✓ scripts/train_baselines.py
✓ dataset/stocknet_final_modeling_set.parquet


## 1. Dataset and Text Audit

We first inspect the existing StockNet modeling dataset and identify the available text/tweet fields.

The dataset contains one row per **ticker × trading day**. The text fields used in this phase are:

- `Company_Texts`
- `Event_Texts`

The initial audit is important because text is sparse: many stock-days contain no associated text.


In [4]:
import pandas as pd

DATA_PATH = "dataset/stocknet_final_modeling_set.parquet"

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns):
    print(f"{i:2d}: {col}")

print("\nFirst 5 rows:")
display(df.head())

Shape: (26603, 45)

Columns:
 0: Date
 1: Open
 2: High
 3: Low
 4: Close
 5: Adj_Close
 6: Volume
 7: Return
 8: Movement_Pct
 9: Target
10: MA_5
11: MA_10
12: MA_20
13: Price_MA5_Ratio
14: Price_MA10_Ratio
15: Price_MA20_Ratio
16: Volatility_5
17: Volatility_20
18: RSI_14
19: MACD
20: MACD_Signal
21: MACD_Hist
22: Volume_Change
23: HL_Spread
24: Ticker
25: Sector
26: Company_Tweet_Count
27: Event_Tweet_Count
28: Total_Tweet_Count
29: Revenue
30: NetIncome
31: TotalAssets
32: TotalLiabilities
33: StockholdersEquity
34: OperatingIncome
35: EPS
36: Cash
37: Profit_Margin
38: Debt_To_Equity
39: ROA
40: Current_Ratio
41: Asset_Turnover
42: Operating_Margin
43: Company_Texts
44: Event_Texts

First 5 rows:


,Date,Open,High,Low,Close,Adj_Close,Volume,Return,Movement_Pct,Target,...,EPS,Cash,Profit_Margin,Debt_To_Equity,ROA,Current_Ratio,Asset_Turnover,Operating_Margin,Company_Texts,Event_Texts
0,2014-01-02,79.382858,79.575714,78.860001,79.018570,73.522530,58671200.0,-0.014064,-1.406374,0,...,8.76,1.425900e+10,0.228633,0.675449,0.039725,0.170867,0.173749,1.36237,$ aapl - wall st . kicks off new year on lower...,$ aapl - chinas xiaomi says to more than doubl...
1,2014-01-03,78.980003,79.099998,77.204285,77.282860,71.907555,98116900.0,-0.021966,-2.196572,0,...,8.76,1.425900e+10,0.228633,0.675449,0.039725,0.170867,0.173749,1.36237,rt AT_USER our top 3 trade ideas for 2014 and ...,overall market vix is down . 35 % where as $ a...
2,2014-01-07,77.760002,77.994286,76.845711,77.148575,71.782608,79302300.0,-0.007151,-0.715129,0,...,8.76,1.425900e+10,0.228633,0.675449,0.039725,0.170867,0.173749,1.36237,$ aapl i love my ipad b / c no virus and i am ...,"if you wanna sell in china , better use chines..."
3,2014-01-08,76.972855,77.937141,76.955711,77.637146,72.237190,64632400.0,0.006333,0.633276,1,...,8.76,1.425900e+10,0.228633,0.675449,0.039725,0.170867,0.173749,1.36237,$ aapl app store sales > $ 10 billion URL digi...,"new deal , new stores : apple ’ s plan for chi..."
4,2014-01-09,78.114288,78.122856,76.478569,76.645714,71.314728,69787200.0,-0.012770,-1.276990,0,...,8.76,1.425900e+10,0.228633,0.675449,0.039725,0.170867,0.173749,1.36237,blackberry's funeral may have to wait . it see...,$ aapl please help me understand the math.bill...


In [5]:
print("Data types:")
display(df.dtypes)

print("\nPotential text/tweet columns:")
text_cols = [
    c for c in df.columns
    if any(word in c.lower() for word in ["tweet", "text", "news", "headline"])
]

print(text_cols)

Data types:


,0
Date,datetime64[ns]
Open,float64
High,float64
Low,float64
Close,float64
Adj_Close,float64
Volume,float64
Return,float64
Movement_Pct,float64
Target,int64



Potential text/tweet columns:
['Company_Tweet_Count', 'Event_Tweet_Count', 'Total_Tweet_Count', 'Company_Texts', 'Event_Texts']


In [6]:
# Show the text columns and a few actual examples

text_cols = [
    c for c in df.columns
    if any(word in c.lower() for word in ["tweet", "text", "news", "headline"])
]

print("Text-related columns:")
print(text_cols)

for col in text_cols:
    print(f"\n{'='*70}")
    print(f"COLUMN: {col}")
    print(f"{'='*70}")

    # Show non-null examples
    examples = df.loc[df[col].notna(), col].head(5)

    for i, value in examples.items():
        print(f"\nRow {i}:")
        print(repr(value))

Text-related columns:
['Company_Tweet_Count', 'Event_Tweet_Count', 'Total_Tweet_Count', 'Company_Texts', 'Event_Texts']

COLUMN: Company_Tweet_Count

Row 0:
32

Row 1:
19

Row 2:
29

Row 3:
9

Row 4:
7

COLUMN: Event_Tweet_Count

Row 0:
1

Row 1:
2

Row 2:
1

Row 3:
2

Row 4:
2

COLUMN: Total_Tweet_Count

Row 0:
33

Row 1:
21

Row 2:
30

Row 3:
11

Row 4:
9

COLUMN: Company_Texts

Row 0:
'$ aapl - wall st . kicks off new year on lower note -> URL stock stocks stockaction [SEP] rt AT_USER here\'s how apple could be making a huge push into healthcare --> URL $ aapl [SEP] rt AT_USER our top 3 trade ideas for 2014 and two $ 1,000 futures wins already in the bank $ gld $ uso $ aapl - - URL [SEP] AT_USER $ aapl having breathing difficulties [SEP] emylers : cheeky\'s board ! : $ aapl up to date company activities and other impo URL [SEP] $ aapl blackberry and singer alicia keys to part ways URL [SEP] rt AT_USER our top 3 trade ideas for 2014 and two $ 1,000 futures wins already in the bank $ 

In [7]:
def count_texts(text):
    if pd.isna(text) or not str(text).strip():
        return 0
    return len(str(text).split("[SEP]"))

for col in ["Company_Texts", "Event_Texts"]:
    counts = df[col].apply(count_texts)

    print(f"\n{col}")
    print("-" * 40)
    print("Rows:", len(counts))
    print("Rows with text:", (counts > 0).sum())
    print("Rows without text:", (counts == 0).sum())
    print("Mean texts/row:", round(counts.mean(), 2))
    print("Median texts/row:", counts.median())
    print("Max texts/row:", counts.max())
    print("\nDistribution:")
    print(counts.describe())


Company_Texts
----------------------------------------
Rows: 26603
Rows with text: 11550
Rows without text: 15053
Mean texts/row: 1.96
Median texts/row: 0.0
Max texts/row: 553

Distribution:
count    26603.000000
mean         1.963914
std          8.175559
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        553.000000
Name: Company_Texts, dtype: float64

Event_Texts
----------------------------------------
Rows: 26603
Rows with text: 2970
Rows without text: 23633
Mean texts/row: 0.19
Median texts/row: 0.0
Max texts/row: 36

Distribution:
count    26603.000000
mean         0.194414
std          0.907961
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         36.000000
Name: Event_Texts, dtype: float64


## 2. Environment and GPU Setup

FinBERT is computationally expensive to run across thousands of stock-days, so the embedding generation is performed with PyTorch on the available CUDA device when possible.

The notebook verifies the PyTorch/CUDA environment before loading the language model.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected — switch Colab Runtime to GPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [10]:
!pip install -q transformers sentencepiece pandas

## 3. Load FinBERT

We use the pretrained **`ProsusAI/finbert`** encoder.

FinBERT is a BERT model adapted to financial-domain language. Its pretrained parameters are used as a frozen feature extractor; we do not fine-tune FinBERT during this baseline experiment.

For each individual text, the representation used here is the 768-dimensional hidden state corresponding to the `[CLS]` token.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading FinBERT...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
finbert = AutoModel.from_pretrained(MODEL_NAME)

finbert = finbert.to(device)
finbert.eval()

print("✓ FinBERT loaded")

Loading FinBERT...
Device: cuda


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ FinBERT loaded


In [7]:
text = "Apple reports strong iPhone sales and investors are optimistic."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = finbert(**inputs)

# CLS token = representation of the whole text
embedding = outputs.last_hidden_state[:, 0, :]

print("Token-level output:", outputs.last_hidden_state.shape)
print("Tweet embedding:", embedding.shape)
print("Embedding dtype:", embedding.dtype)
print("Embedding device:", embedding.device)

Token-level output: torch.Size([1, 12, 768])
Tweet embedding: torch.Size([1, 768])
Embedding dtype: torch.float32
Embedding device: cuda:0


## 4. Text → Daily 768-Dimensional Embeddings

Each stock-day can contain multiple `[SEP]`-separated texts.

For a stock-day with multiple texts, the pipeline is:

```text
Text 1 ─┐
Text 2 ─┤
Text 3 ─┤→ FinBERT → 768-D embedding each → mean pooling → one daily embedding
  ... ──┘
```

For a stock-day with no text, the embedding array remains a zero vector.

This gives every stock-day a fixed-size 768-dimensional representation that can be aligned directly with the original modeling rows.


In [8]:
def embed_texts(texts, batch_size=32, max_length=128):
    """
    Convert a list of texts into FinBERT [CLS] embeddings.

    Returns:
        Tensor of shape [len(texts), 768]
    """
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = finbert(**inputs)

        # [CLS] embedding
        batch_embeddings = outputs.last_hidden_state[:, 0, :]

        # Move off GPU immediately
        all_embeddings.append(batch_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)


def get_daily_embedding(text):
    """
    Convert all [SEP]-separated texts in one stock-day
    into one 768-dimensional mean embedding.
    """
    if pd.isna(text) or not str(text).strip():
        return np.zeros(768, dtype=np.float32)

    texts = [
        t.strip()
        for t in str(text).split("[SEP]")
        if t.strip()
    ]

    if not texts:
        return np.zeros(768, dtype=np.float32)

    embeddings = embed_texts(texts, batch_size=32)

    return embeddings.mean(dim=0).numpy().astype(np.float32)

In [12]:
import numpy as np
import pandas as pd

DATA_PATH = "dataset/stocknet_final_modeling_set.parquet"
df = pd.read_parquet(DATA_PATH)

test_text = df.loc[0, "Company_Texts"]

print("Number of tweets:", len(test_text.split("[SEP]")))

daily_embedding = get_daily_embedding(test_text)

print("Daily embedding shape:", daily_embedding.shape)
print("dtype:", daily_embedding.dtype)
print("First 10 values:", daily_embedding[:10])
print("L2 norm:", np.linalg.norm(daily_embedding))

Number of tweets: 32
Daily embedding shape: (768,)
dtype: float32
First 10 values: [ 0.16803662  0.1516009  -0.19392832 -0.43315485 -0.01998157 -0.38601243
  0.39926407  0.549522    0.37453502 -0.29268807]
L2 norm: 12.734047


## 5. Generate Company and Event Embeddings

The embedding-generation function processes each non-empty stock-day text field and stores one 768-dimensional vector per original dataframe row.

Two arrays are generated:

- `company_embeddings.npy`
- `event_embeddings.npy`

The arrays retain the exact row ordering of the original dataframe so that each embedding can be joined back to the corresponding ticker/date row without changing the dataset population.


In [13]:
import time
import numpy as np

# ---------------------------------------------------------
# Generate daily embeddings for one text column
# ---------------------------------------------------------
def generate_embeddings_for_column(df, column, output_path):
    embeddings = np.zeros((len(df), 768), dtype=np.float32)

    non_empty = df[column].notna() & (
        df[column].astype(str).str.strip() != ""
    )

    indices = df.index[non_empty]

    print(f"\n{'='*70}")
    print(f"Generating embeddings: {column}")
    print(f"Rows with text: {len(indices)}")
    print(f"Total rows: {len(df)}")
    print(f"{'='*70}")

    start_time = time.time()

    for count, idx in enumerate(indices, 1):

        embeddings[idx] = get_daily_embedding(
            df.loc[idx, column]
        )

        if count % 100 == 0 or count == len(indices):
            elapsed = time.time() - start_time
            avg = elapsed / count
            remaining = avg * (len(indices) - count)

            print(
                f"{count:,}/{len(indices):,} | "
                f"elapsed: {elapsed/60:.1f} min | "
                f"ETA: {remaining/60:.1f} min"
            )

    np.save(output_path, embeddings)

    elapsed = time.time() - start_time

    print(f"\n✓ Finished {column}")
    print(f"Time: {elapsed/60:.2f} minutes")
    print(f"Shape: {embeddings.shape}")
    print(f"Saved to: {output_path}")

    return embeddings

In [14]:
company_embeddings = generate_embeddings_for_column(
    df,
    "Company_Texts",
    "company_embeddings.npy"
)


Generating embeddings: Company_Texts
Rows with text: 11550
Total rows: 26603
100/11,550 | elapsed: 0.1 min | ETA: 16.0 min
200/11,550 | elapsed: 0.3 min | ETA: 15.7 min
300/11,550 | elapsed: 0.4 min | ETA: 16.3 min
400/11,550 | elapsed: 0.5 min | ETA: 13.2 min
500/11,550 | elapsed: 0.5 min | ETA: 10.9 min
600/11,550 | elapsed: 0.5 min | ETA: 9.3 min
700/11,550 | elapsed: 0.5 min | ETA: 8.1 min
800/11,550 | elapsed: 0.5 min | ETA: 7.3 min
900/11,550 | elapsed: 0.6 min | ETA: 7.1 min
1,000/11,550 | elapsed: 0.6 min | ETA: 6.8 min
1,100/11,550 | elapsed: 0.7 min | ETA: 6.5 min
1,200/11,550 | elapsed: 0.7 min | ETA: 6.3 min
1,300/11,550 | elapsed: 0.7 min | ETA: 5.9 min
1,400/11,550 | elapsed: 0.8 min | ETA: 5.6 min
1,500/11,550 | elapsed: 0.8 min | ETA: 5.4 min
1,600/11,550 | elapsed: 0.8 min | ETA: 5.2 min
1,700/11,550 | elapsed: 0.9 min | ETA: 5.0 min
1,800/11,550 | elapsed: 0.9 min | ETA: 4.8 min
1,900/11,550 | elapsed: 0.9 min | ETA: 4.6 min
2,000/11,550 | elapsed: 0.9 min | ETA: 4.4

In [15]:
event_embeddings = generate_embeddings_for_column(
    df,
    "Event_Texts",
    "event_embeddings.npy"
)


Generating embeddings: Event_Texts
Rows with text: 2970
Total rows: 26603
100/2,970 | elapsed: 0.1 min | ETA: 1.5 min
200/2,970 | elapsed: 0.1 min | ETA: 1.1 min
300/2,970 | elapsed: 0.1 min | ETA: 1.0 min
400/2,970 | elapsed: 0.1 min | ETA: 0.9 min
500/2,970 | elapsed: 0.2 min | ETA: 0.8 min
600/2,970 | elapsed: 0.2 min | ETA: 0.7 min
700/2,970 | elapsed: 0.2 min | ETA: 0.6 min
800/2,970 | elapsed: 0.2 min | ETA: 0.5 min
900/2,970 | elapsed: 0.2 min | ETA: 0.5 min
1,000/2,970 | elapsed: 0.2 min | ETA: 0.5 min
1,100/2,970 | elapsed: 0.3 min | ETA: 0.4 min
1,200/2,970 | elapsed: 0.3 min | ETA: 0.4 min
1,300/2,970 | elapsed: 0.3 min | ETA: 0.4 min
1,400/2,970 | elapsed: 0.3 min | ETA: 0.3 min
1,500/2,970 | elapsed: 0.3 min | ETA: 0.3 min
1,600/2,970 | elapsed: 0.3 min | ETA: 0.3 min
1,700/2,970 | elapsed: 0.3 min | ETA: 0.3 min
1,800/2,970 | elapsed: 0.4 min | ETA: 0.2 min
1,900/2,970 | elapsed: 0.4 min | ETA: 0.2 min
2,000/2,970 | elapsed: 0.4 min | ETA: 0.2 min
2,100/2,970 | elapsed: 

### Embedding Availability Check

The zero-vector counts provide a useful sanity check for missing text.

From the text audit:

- Company text is present for **11,550 / 26,603** rows.
- Company text is absent for **15,053 / 26,603** rows.
- Event text is present for **2,970 / 26,603** rows.
- Event text is absent for **23,633 / 26,603** rows.

The absence of text is retained rather than dropping those stock-days.


In [16]:
import numpy as np
import os

company_embeddings = np.load("company_embeddings.npy")
event_embeddings = np.load("event_embeddings.npy")

print("Company:", company_embeddings.shape)
print("Event:", event_embeddings.shape)

print("\nCompany zero-vector rows:",
      np.sum(np.linalg.norm(company_embeddings, axis=1) == 0))

print("Event zero-vector rows:",
      np.sum(np.linalg.norm(event_embeddings, axis=1) == 0))

print("\nFiles:")
print("company_embeddings.npy:", os.path.getsize("company_embeddings.npy") / 1024**2, "MB")
print("event_embeddings.npy:", os.path.getsize("event_embeddings.npy") / 1024**2, "MB")

Company: (26603, 768)
Event: (26603, 768)

Company zero-vector rows: 15053
Event zero-vector rows: 23633

Files:
company_embeddings.npy: 77.9385986328125 MB
event_embeddings.npy: 77.9385986328125 MB


In [18]:
print("DataFrame shape:", df.shape)

print("\nFirst 5 indices:")
print(df.index[:5].tolist())

print("\nLast 5 indices:")
print(df.index[-5:].tolist())

print("\nIndex unique:", df.index.is_unique)
print("Index starts at 0:", df.index.min() == 0)
print("Index ends at len(df)-1:", df.index.max() == len(df)-1)

DataFrame shape: (26603, 45)

First 5 indices:
[0, 1, 2, 3, 4]

Last 5 indices:
[26598, 26599, 26600, 26601, 26602]

Index unique: True
Index starts at 0: True
Index ends at len(df)-1: True


In [19]:
print("All 45 columns:\n")

for i, col in enumerate(df.columns):
    print(f"{i:2d}: {col}")

All 45 columns:

 0: Date
 1: Open
 2: High
 3: Low
 4: Close
 5: Adj_Close
 6: Volume
 7: Return
 8: Movement_Pct
 9: Target
10: MA_5
11: MA_10
12: MA_20
13: Price_MA5_Ratio
14: Price_MA10_Ratio
15: Price_MA20_Ratio
16: Volatility_5
17: Volatility_20
18: RSI_14
19: MACD
20: MACD_Signal
21: MACD_Hist
22: Volume_Change
23: HL_Spread
24: Ticker
25: Sector
26: Company_Tweet_Count
27: Event_Tweet_Count
28: Total_Tweet_Count
29: Revenue
30: NetIncome
31: TotalAssets
32: TotalLiabilities
33: StockholdersEquity
34: OperatingIncome
35: EPS
36: Cash
37: Profit_Margin
38: Debt_To_Equity
39: ROA
40: Current_Ratio
41: Asset_Turnover
42: Operating_Margin
43: Company_Texts
44: Event_Texts


In [20]:
print("\nColumns that look like target/date/ticker identifiers:")

for col in df.columns:
    name = col.lower()
    if any(x in name for x in [
        "target", "label", "return", "date", "ticker",
        "symbol", "stock", "company"
    ]):
        print(col)


Columns that look like target/date/ticker identifiers:
Date
Return
Target
Ticker
Company_Tweet_Count
StockholdersEquity
Company_Texts


## 6. Existing Phase 1 Feature Definitions

Before adding embeddings, we inspect the original feature sets.

Phase 1 contains four structured feature sets:

- **FS1:** Price / technical features
- **FS2:** Price + Fundamentals
- **FS3:** Price + Tweet Counts
- **FS4:** Price + Fundamentals + Tweet Counts

The original feature definitions are kept unchanged. Phase 2 adds new feature sets rather than modifying the Phase 1 definitions, allowing direct comparison between the experiments.


In [21]:
from src.data.stocknet_dataset import FEATURE_SETS

import pprint

pprint.pp(FEATURE_SETS)

{'FS1_Price': ['Return',
               'RSI_14',
               'MACD',
               'MACD_Signal',
               'MACD_Hist',
               'Volatility_5',
               'Volatility_20',
               'Price_MA5_Ratio',
               'Price_MA10_Ratio',
               'Price_MA20_Ratio',
               'Volume_Change',
               'HL_Spread',
               'MA_5',
               'MA_10'],
 'FS2_Price_Fundamentals': ['Return',
                            'RSI_14',
                            'MACD',
                            'MACD_Signal',
                            'MACD_Hist',
                            'Volatility_5',
                            'Volatility_20',
                            'Price_MA5_Ratio',
                            'Price_MA10_Ratio',
                            'Price_MA20_Ratio',
                            'Volume_Change',
                            'HL_Spread',
                            'MA_5',
                            'MA_10',
       

## 7. Existing Dataset Pipeline

The original dataset pipeline performs:

1. Data loading and cleaning
2. Global chronological train/validation/test splitting
3. Normalization using training-set statistics only
4. Sliding-window construction

Phase 2 reuses this protocol so that the main experimental difference is the feature representation rather than the split or preprocessing strategy.


In [22]:
import inspect
from src.data import stocknet_dataset

print(inspect.getsource(stocknet_dataset.build_datasets))

def build_datasets(parquet_path: str, feature_set: str = 'FS1_Price',
                   window_size: int = 5,
                   ) -> Tuple[StockNetDataset, StockNetDataset, StockNetDataset, dict]:
    """
    Full pipeline:
      load -> clean -> global date split -> normalize -> build datasets.

    Returns (train_dataset, val_dataset, test_dataset, info_dict).
    Split boundaries (literature-standard):
      Train : 2014-01-02 -> 2015-03-31
      Val   : 2015-04-01 -> 2015-07-31
      Test  : 2015-08-01 -> 2015-12-31
    """
    assert feature_set in FEATURE_SETS, \
        f"feature_set must be one of {list(FEATURE_SETS.keys())}"

    feature_cols = FEATURE_SETS[feature_set]
    df = load_and_clean(parquet_path)

    train_df, val_df, test_df = split_by_date(df)

    # Normalize using train stats only -- no leakage into val/test
    means, stds = compute_norm_stats(train_df, feature_cols)
    train_df = normalize(train_df, feature_cols, means, stds)
    val_df   = normalize(val_d

In [23]:
import inspect
from src.data import stocknet_dataset

print(inspect.getsource(stocknet_dataset.StockNetDataset))

class StockNetDataset(Dataset):
    """
    PyTorch Dataset for MMGTFFF.

    Builds sliding windows of W days per ticker.
    feature_set selects which columns to use:
        'FS1_Price', 'FS2_Price_Fundamentals',
        'FS3_Price_Tweets', 'FS4_Full_Structured'
    """

    def __init__(self, df: pd.DataFrame, feature_cols: List[str],
                 window_size: int = 5):
        self.feature_cols = feature_cols
        self.window_size  = window_size
        self.samples      = []
        self._build(df)

    def _build(self, df: pd.DataFrame):
        for ticker, group in df.groupby('Ticker'):
            group  = group.sort_values('Date').reset_index(drop=True)
            feats  = group[self.feature_cols].values.astype(np.float32)
            labels = group['Target'].values.astype(np.int64)
            dates  = group['Date'].values
            sids   = group['Sector_ID'].values.astype(np.int64)
            ctexts = group['Company_Texts'].values
            etexts = group['Eve

## 8. Construct the Phase 2 Modeling Dataset

The original modeling dataframe is copied rather than modified in place.

The company embedding representation adds:

- 768 FinBERT dimensions: `CompanyEmb_000` … `CompanyEmb_767`
- `Company_Text_Present`: an explicit indicator for whether company text exists for that stock-day

The original 26,603 rows are retained.


In [24]:
df_phase2 = df.copy()

In [25]:
# Load embeddings
company_embeddings = np.load("company_embeddings.npy")

print("Embedding shape:", company_embeddings.shape)

# Safety check
assert len(df) == len(company_embeddings), (
    f"Row mismatch: df={len(df)}, embeddings={len(company_embeddings)}"
)

# Create a separate Phase 2 dataframe
df_phase2 = df.copy()

# Add presence indicator
df_phase2["Company_Text_Present"] = (
    df_phase2["Company_Texts"].notna() &
    (df_phase2["Company_Texts"].astype(str).str.strip() != "")
).astype(np.float32)

# Add 768 embedding columns
company_embedding_cols = [
    f"CompanyEmb_{i:03d}" for i in range(768)
]

embedding_df = pd.DataFrame(
    company_embeddings,
    columns=company_embedding_cols,
    index=df_phase2.index
)

df_phase2 = pd.concat(
    [df_phase2, embedding_df],
    axis=1
)

print("Original shape:", df.shape)
print("Phase 2 shape:", df_phase2.shape)

print("\nNew columns:")
print(company_embedding_cols[:5], "...", company_embedding_cols[-5:])
print("Company_Text_Present:", df_phase2["Company_Text_Present"].value_counts().to_dict())

Embedding shape: (26603, 768)
Original shape: (26603, 45)
Phase 2 shape: (26603, 814)

New columns:
['CompanyEmb_000', 'CompanyEmb_001', 'CompanyEmb_002', 'CompanyEmb_003', 'CompanyEmb_004'] ... ['CompanyEmb_763', 'CompanyEmb_764', 'CompanyEmb_765', 'CompanyEmb_766', 'CompanyEmb_767']
Company_Text_Present: {0.0: 15053, 1.0: 11550}


In [26]:
PHASE2_PATH = "dataset/stocknet_final_modeling_set_phase2.parquet"

df_phase2.to_parquet(PHASE2_PATH, index=False)

print("Saved:", PHASE2_PATH)

# Verify
test_df = pd.read_parquet(PHASE2_PATH)

print("Reloaded shape:", test_df.shape)

embedding_cols = [
    c for c in test_df.columns
    if c.startswith("CompanyEmb_")
]

print("Embedding columns:", len(embedding_cols))
print("Has presence flag:", "Company_Text_Present" in test_df.columns)

Saved: dataset/stocknet_final_modeling_set_phase2.parquet
Reloaded shape: (26603, 814)
Embedding columns: 768
Has presence flag: True


## 9. Phase 2 Feature Sets

The first Phase 2 experiments focus on **Company Text embeddings**.

### FS5 — Price + Company FinBERT

- 14 price/technical features
- 768 company embedding dimensions
- 1 text-presence indicator
- **783 features total**

### FS6 — Price + Fundamentals + Company FinBERT

- 14 price/technical features
- 8 fundamental features
- 768 company embedding dimensions
- 1 text-presence indicator
- **791 features total**

The existing Phase 1 feature sets remain untouched.


In [27]:
from src.data.stocknet_dataset import FEATURE_SETS

# Keep the original Phase 1 feature sets untouched
PRICE_FEATURES = FEATURE_SETS["FS1_Price"]
FUNDAMENTAL_FEATURES = [
    "Revenue",
    "NetIncome",
    "TotalAssets",
    "TotalLiabilities",
    "StockholdersEquity",
    "EPS",
    "Cash",
    "ROA",
]

COMPANY_EMBEDDING_FEATURES = [
    f"CompanyEmb_{i:03d}" for i in range(768)
]

# Phase 2 feature sets
PHASE2_FEATURE_SETS = {
    "FS5_Price_CompanyEmbedding":
        PRICE_FEATURES
        + COMPANY_EMBEDDING_FEATURES
        + ["Company_Text_Present"],

    "FS6_Price_Fundamentals_CompanyEmbedding":
        PRICE_FEATURES
        + FUNDAMENTAL_FEATURES
        + COMPANY_EMBEDDING_FEATURES
        + ["Company_Text_Present"],
}

for name, features in PHASE2_FEATURE_SETS.items():
    print(name)
    print("Number of features:", len(features))
    print("First 5:", features[:5])
    print("Last 5:", features[-5:])
    print()

FS5_Price_CompanyEmbedding
Number of features: 783
First 5: ['Return', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']
Last 5: ['CompanyEmb_764', 'CompanyEmb_765', 'CompanyEmb_766', 'CompanyEmb_767', 'Company_Text_Present']

FS6_Price_Fundamentals_CompanyEmbedding
Number of features: 791
First 5: ['Return', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist']
Last 5: ['CompanyEmb_764', 'CompanyEmb_765', 'CompanyEmb_766', 'CompanyEmb_767', 'Company_Text_Present']



In [28]:
for name, features in PHASE2_FEATURE_SETS.items():
    missing = [f for f in features if f not in df_phase2.columns]

    print(
        f"{name}: "
        f"{len(features)} requested, "
        f"{len(missing)} missing"
    )

    if missing:
        print("Missing:", missing[:20])

FS5_Price_CompanyEmbedding: 783 requested, 0 missing
FS6_Price_Fundamentals_CompanyEmbedding: 791 requested, 0 missing


## 10. Phase 2 Data Pipeline

The Phase 2 builder follows the same preprocessing protocol as Phase 1:

```text
Phase 2 Parquet
      ↓
original data cleaning
      ↓
global chronological split
      ↓
train-only normalization
      ↓
5-day sliding windows
      ↓
model-ready tensors
```

The embedding dimensions are treated as ordinary numerical features by the existing `StockNetDataset`.


In [44]:
import pandas as pd
import numpy as np

from src.data.stocknet_dataset import (
    StockNetDataset,
    load_and_clean,
    split_by_date,
    compute_norm_stats,
    normalize,
    TRAIN_END,
    VAL_START,
    VAL_END,
)

def build_phase2_datasets(
    parquet_path,
    feature_set,
    feature_sets,
    window_size=5
):
    """
    Phase 2 dataset pipeline.

    Same protocol as Phase 1:
      load
      -> chronological split
      -> train-only normalization
      -> sliding windows

    The only difference is that the dataframe contains
    the fixed FinBERT embeddings.
    """

    assert feature_set in feature_sets, (
        f"Unknown feature set: {feature_set}"
    )

    feature_cols = feature_sets[feature_set]

    # Load Phase 2 dataframe
    df = load_and_clean(parquet_path)

    # Make sure dates are datetime
    df["Date"] = pd.to_datetime(df["Date"])

    # Chronological split
    train_df, val_df, test_df = split_by_date(df)

    print("Split sizes:")
    print("  Train rows:", len(train_df))
    print("  Val rows:  ", len(val_df))
    print("  Test rows: ", len(test_df))

    # Train-only normalization
    means, stds = compute_norm_stats(
        train_df,
        feature_cols
    )

    train_df = normalize(
        train_df,
        feature_cols,
        means,
        stds
    )

    val_df = normalize(
        val_df,
        feature_cols,
        means,
        stds
    )

    test_df = normalize(
        test_df,
        feature_cols,
        means,
        stds
    )

    # Build sliding-window datasets
    train_ds = StockNetDataset(
    train_df,
    feature_cols,
    window_size
    )

    val_ds = StockNetDataset(
        val_df,
        feature_cols,
        window_size
    )

    test_ds = StockNetDataset(
        test_df,
        feature_cols,
        window_size
    )

    info = {
        "feature_set": feature_set,
        "feature_cols": feature_cols,
        "num_features": len(feature_cols),
        "window_size": window_size,
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": len(test_ds),
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
    }

    return train_ds, val_ds, test_ds, info

In [31]:
original_df = pd.read_parquet(
    "dataset/stocknet_final_modeling_set.parquet"
)

print("Original shape:", original_df.shape)

print("Sector_ID exists:",
      "Sector_ID" in original_df.columns)

if "Sector_ID" in original_df.columns:
    print(original_df["Sector_ID"].head())
    print(original_df["Sector_ID"].nunique(), "unique Sector_IDs")

Original shape: (26603, 45)
Sector_ID exists: False


In [33]:
from torch.utils.data import Dataset
import torch
import numpy as np
import pandas as pd


class StockNetDatasetPhase2(Dataset):
    """
    Same behavior as the original StockNetDataset,
    but does not require Sector_ID.

    Used for the Phase 2 embedding experiments.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols,
        window_size=5
    ):
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.samples = []
        self._build(df)

    def _build(self, df):

        for ticker, group in df.groupby("Ticker"):

            group = (
                group
                .sort_values("Date")
                .reset_index(drop=True)
            )

            feats = group[self.feature_cols].values.astype(
                np.float32
            )

            labels = group["Target"].values.astype(
                np.int64
            )

            dates = group["Date"].values

            # Sector_ID isn't present in this dataset.
            # Keep None for compatibility with metadata.
            has_sector = "Sector_ID" in group.columns

            if has_sector:
                sids = group["Sector_ID"].values.astype(
                    np.int64
                )

            ctexts = group["Company_Texts"].values
            etexts = group["Event_Texts"].values

            for i in range(
                self.window_size,
                len(group)
            ):

                self.samples.append({

                    # Previous W days
                    "window": feats[
                        i - self.window_size:i
                    ],

                    # Yesterday's features
                    "flat": feats[i - 1],

                    # Today's target
                    "target": labels[i],

                    "ticker": ticker,
                    "date": dates[i],

                    "sector_id":
                        int(sids[i])
                        if has_sector
                        else -1,

                    "company_text": ctexts[i],
                    "event_text": etexts[i],
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        s = self.samples[idx]

        return {
            "window": torch.tensor(
                s["window"],
                dtype=torch.float32
            ),

            "flat": torch.tensor(
                s["flat"],
                dtype=torch.float32
            ),

            "target": torch.tensor(
                s["target"],
                dtype=torch.long
            ),

            "sector_id": torch.tensor(
                s["sector_id"],
                dtype=torch.long
            ),
        }

    def get_metadata(self, idx):

        s = self.samples[idx]

        return {
            k: s[k]
            for k in (
                "ticker",
                "date",
                "company_text",
                "event_text"
            )
        }

    def to_numpy(self, use_window=False):

        X = np.array([
            s["window"].flatten()
            if use_window
            else s["flat"]
            for s in self.samples
        ])

        y = np.array([
            s["target"]
            for s in self.samples
        ])

        return X, y

### Phase 2 Dataset Sanity Check

The expected model input for FS5 is:

```text
5 days × 783 features
```

The target remains the existing binary next-day direction label.

A NaN check is performed before training. The original cleaning routine handles the isolated missing values in the existing technical features.


In [45]:
train_ds, val_ds, test_ds, info = build_phase2_datasets(
    PHASE2_PATH,
    "FS5_Price_CompanyEmbedding",
    PHASE2_FEATURE_SETS,
    window_size=5
)

print(info)

sample = train_ds[0]

print("\nWindow shape:", sample["window"].shape)
print("Flat shape:", sample["flat"].shape)
print("Target:", sample["target"].item())

print("\nNaN check:")
print("NaNs in window:", torch.isnan(sample["window"]).sum().item())
print("NaNs in flat:", torch.isnan(sample["flat"]).sum().item())

Split sizes:
  Train rows: 15969
  Val rows:   4359
  Test rows:  6275
{'feature_set': 'FS5_Price_CompanyEmbedding', 'feature_cols': ['Return', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Volatility_5', 'Volatility_20', 'Price_MA5_Ratio', 'Price_MA10_Ratio', 'Price_MA20_Ratio', 'Volume_Change', 'HL_Spread', 'MA_5', 'MA_10', 'CompanyEmb_000', 'CompanyEmb_001', 'CompanyEmb_002', 'CompanyEmb_003', 'CompanyEmb_004', 'CompanyEmb_005', 'CompanyEmb_006', 'CompanyEmb_007', 'CompanyEmb_008', 'CompanyEmb_009', 'CompanyEmb_010', 'CompanyEmb_011', 'CompanyEmb_012', 'CompanyEmb_013', 'CompanyEmb_014', 'CompanyEmb_015', 'CompanyEmb_016', 'CompanyEmb_017', 'CompanyEmb_018', 'CompanyEmb_019', 'CompanyEmb_020', 'CompanyEmb_021', 'CompanyEmb_022', 'CompanyEmb_023', 'CompanyEmb_024', 'CompanyEmb_025', 'CompanyEmb_026', 'CompanyEmb_027', 'CompanyEmb_028', 'CompanyEmb_029', 'CompanyEmb_030', 'CompanyEmb_031', 'CompanyEmb_032', 'CompanyEmb_033', 'CompanyEmb_034', 'CompanyEmb_035', 'CompanyEmb_036', 'Comp

In [37]:
print("=== Phase 2 Dataset ===")
print("Train samples:", len(train_ds))
print("Val samples:  ", len(val_ds))
print("Test samples: ", len(test_ds))
print("Features:     ", info["num_features"])
print("Window size:  ", info["window_size"])

print("\n=== Sample ===")
sample = train_ds[0]

print("Window shape:", sample["window"].shape)
print("Flat shape:  ", sample["flat"].shape)
print("Target:      ", sample["target"].item())

print("\n=== Feature range of first sample ===")
print("Min:", sample["window"].min().item())
print("Max:", sample["window"].max().item())
print("Mean:", sample["window"].mean().item())
print("Std:", sample["window"].std().item())

=== Phase 2 Dataset ===
Train samples: 15534
Val samples:   3924
Test samples:  5840
Features:      783
Window size:   5

=== Sample ===
Window shape: torch.Size([5, 783])
Flat shape:   torch.Size([783])
Target:       0

=== Feature range of first sample ===
Min: nan
Max: nan
Mean: nan
Std: nan


In [38]:
# Check the Phase 2 dataframe BEFORE normalization

feature_cols = PHASE2_FEATURE_SETS[
    "FS5_Price_CompanyEmbedding"
]

nan_counts = df_phase2[feature_cols].isna().sum()

nan_features = nan_counts[nan_counts > 0].sort_values(
    ascending=False
)

print("Number of features containing NaNs:",
      len(nan_features))

print("\nFeatures with NaNs:")
print(nan_features.head(30))

Number of features containing NaNs: 2

Features with NaNs:
Volatility_20    1
Volume_Change    1
dtype: int64


In [39]:
from src.data.stocknet_dataset import load_and_clean
import inspect

print(inspect.getsource(load_and_clean))

def load_and_clean(parquet_path: str) -> pd.DataFrame:
    """Load parquet and fix known data quality issues."""
    df = pd.read_parquet(parquet_path)
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    # Fix infinities in Volume_Change
    df['Volume_Change'] = df['Volume_Change'].replace([np.inf, -np.inf], 0.0)

    # Fill remaining nulls in price/technical features
    for col in PRICE_FEATURES + TWEET_FEATURES:
        if col in df.columns and df[col].isnull().any():
            df[col] = df[col].fillna(0.0)

    # Forward-fill fundamentals within each ticker, then zero-fill
    for col in ALL_FUNDAMENTAL_COLS:
        if col in df.columns:
            df[col] = df.groupby('Ticker')[col].ffill()
            df[col] = df[col].fillna(0.0)

    df['Sector_ID'] = df['Sector'].map(SECTOR_MAP).fillna(0).astype(int)

    return df



In [40]:
from src.data.stocknet_dataset import load_and_clean

clean_df = load_and_clean(PHASE2_PATH)

print("Cleaned shape:", clean_df.shape)
print("Sector_ID exists:", "Sector_ID" in clean_df.columns)

feature_cols = PHASE2_FEATURE_SETS["FS5_Price_CompanyEmbedding"]

nan_counts = clean_df[feature_cols].isna().sum()
print("\nRemaining NaNs:")
print(nan_counts[nan_counts > 0])

Cleaned shape: (26603, 815)
Sector_ID exists: True

Remaining NaNs:
Series([], dtype: int64)


## 11. Model Architectures

Phase 2 uses the same baseline architectures as Phase 1:

### Logistic Regression
Operates on the previous day's flattened feature vector.

### LSTM
Receives the 5-day sequential window:

```text
5 × feature_dimension
```

with:

- 2 LSTM layers
- hidden dimension = 64
- dropout = 0.2

### MLP
Operates on the previous day's flattened feature vector with:

- hidden dimension = 128
- dropout = 0.2

This keeps the model architecture fixed while changing the feature representation.


In [46]:
import inspect
from src.models.baselines import MLPBaseline, LSTMBaseline

print("MLP:")
print(inspect.getsource(MLPBaseline))

print("\nLSTM:")
print(inspect.getsource(LSTMBaseline))

MLP:
class MLPBaseline(nn.Module):
    """MLP on flat feature vector (current day)."""

    def __init__(self, input_dim: int, hidden_dim: int = 128,
                 dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),        nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, flat: torch.Tensor, **kwargs) -> torch.Tensor:
        return self.net(flat)


LSTM:
class LSTMBaseline(nn.Module):
    """LSTM with sliding window — sequential model."""

    def __init__(self, input_dim: int, hidden_dim: int = 64,
                 num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
   

## 12. Phase 2 Experiment — 2 Feature Sets × 3 Models

The complete baseline grid is run in one experiment:

| Feature set | Logistic Regression | LSTM | MLP |
|---|---|---|---|
| FS5 — Price + Company FinBERT | ✓ | ✓ | ✓ |
| FS6 — Price + Fundamentals + Company FinBERT | ✓ | ✓ | ✓ |

Training configuration:

- Seed = 42
- Window size = 5
- Maximum epochs = 50
- Early-stopping patience = 10
- Learning rate = 0.001

Metrics:

- Accuracy
- F1
- MCC
- ROC-AUC


In [47]:
import os
import json
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression

from src.models.baselines import LSTMBaseline, MLPBaseline
from src.training.trainer import Trainer, compute_metrics
from src.utils.seed import set_seed


# ============================================================
# PHASE 2 — FINBERT EMBEDDING BASELINES
#
# 2 Feature Sets × 3 Models = 6 Experiments
#
# FS5: Price + Company FinBERT Embedding
# FS6: Price + Fundamentals + Company FinBERT Embedding
#
# Models:
#   - Logistic Regression
#   - LSTM
#   - MLP
#
# Same protocol as Phase 1:
#   Seed       = 42
#   Window     = 5
#   Epochs     = 50
#   Patience   = 10
#   LR         = 0.001
# ============================================================


PHASE2_PATH = "dataset/stocknet_final_modeling_set_phase2.parquet"

SEED = 42
EPOCHS = 50
PATIENCE = 10
WINDOW_SIZE = 5


# ------------------------------------------------------------
# Helper: Logistic Regression
# ------------------------------------------------------------

def run_logistic_regression(train_ds, test_ds, save_dir):

    X_train, y_train = train_ds.to_numpy(
        use_window=False
    )

    X_test, y_test = test_ds.to_numpy(
        use_window=False
    )

    print(
        f"      LR input shape: "
        f"train={X_train.shape}, test={X_test.shape}"
    )

    model = LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        C=1.0
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    probs = model.predict_proba(X_test)[:, 1]

    metrics = compute_metrics(
        y_test,
        preds,
        probs
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    with open(
        os.path.join(save_dir, "metrics.json"),
        "w"
    ) as f:
        json.dump(
            {"test": metrics},
            f,
            indent=2
        )

    return metrics


# ------------------------------------------------------------
# Helper: LSTM
# ------------------------------------------------------------

def run_lstm(
    train_ds,
    val_ds,
    test_ds,
    input_dim,
    save_dir
):

    set_seed(SEED)

    model = LSTMBaseline(
        input_dim=input_dim,
        hidden_dim=64,
        num_layers=2,
        dropout=0.2
    )

    trainer = Trainer(
        model,
        train_ds,
        val_ds,
        test_ds,
        lr=0.001,
        save_dir=save_dir
    )

    return trainer.train(
        num_epochs=EPOCHS,
        patience=PATIENCE
    )


# ------------------------------------------------------------
# Helper: MLP
# ------------------------------------------------------------

def run_mlp(
    train_ds,
    val_ds,
    test_ds,
    input_dim,
    save_dir
):

    set_seed(SEED)

    model = MLPBaseline(
        input_dim=input_dim,
        hidden_dim=128,
        dropout=0.2
    )

    trainer = Trainer(
        model,
        train_ds,
        val_ds,
        test_ds,
        lr=0.001,
        save_dir=save_dir
    )

    return trainer.train(
        num_epochs=EPOCHS,
        patience=PATIENCE
    )


# ------------------------------------------------------------
# Helper: formatting
# ------------------------------------------------------------

def fmt(m):

    return (
        f"Acc={m['accuracy']:.4f}  "
        f"F1={m['f1']:.4f}  "
        f"MCC={m['mcc']:.4f}  "
        f"AUC={m.get('auc', 0.5):.4f}"
    )


# ============================================================
# MAIN EXPERIMENT
# ============================================================

def run_phase2():

    set_seed(SEED)

    all_results = {}

    feature_set_labels = {

        "FS5_Price_CompanyEmbedding":
            "Price + Company FinBERT",

        "FS6_Price_Fundamentals_CompanyEmbedding":
            "Price + Fundamentals + Company FinBERT",
    }


    for fs_key, fs_label in feature_set_labels.items():

        print("\n")
        print("=" * 80)
        print(f"  Feature Set: {fs_label}")
        print("=" * 80)


        # ----------------------------------------------------
        # Build datasets
        # ----------------------------------------------------

        train_ds, val_ds, test_ds, info = (
            build_phase2_datasets(
                PHASE2_PATH,
                fs_key,
                PHASE2_FEATURE_SETS,
                window_size=WINDOW_SIZE
            )
        )

        n_feat = info["num_features"]

        print(
            f"\n  Features = {n_feat}"
        )

        print(
            f"  Train = {info['train_size']} | "
            f"Val = {info['val_size']} | "
            f"Test = {info['test_size']}"
        )


        fs_results = {}


        # ----------------------------------------------------
        # 1. Logistic Regression
        # ----------------------------------------------------

        print(
            "\n  [1/3] Logistic Regression"
        )

        save_dir = (
            f"results/phase2_embeddings/"
            f"{fs_key}/logistic_regression"
        )

        metrics = run_logistic_regression(
            train_ds,
            test_ds,
            save_dir
        )

        fs_results[
            "Logistic Regression"
        ] = metrics

        print(
            f"    {fmt(metrics)}"
        )


        # ----------------------------------------------------
        # 2. LSTM
        # ----------------------------------------------------

        print(
            "\n  [2/3] LSTM (sliding window, W=5)"
        )

        save_dir = (
            f"results/phase2_embeddings/"
            f"{fs_key}/lstm"
        )

        metrics = run_lstm(
            train_ds,
            val_ds,
            test_ds,
            n_feat,
            save_dir
        )

        fs_results["LSTM"] = metrics

        print(
            f"    {fmt(metrics)}"
        )


        # ----------------------------------------------------
        # 3. MLP
        # ----------------------------------------------------

        print(
            "\n  [3/3] MLP"
        )

        save_dir = (
            f"results/phase2_embeddings/"
            f"{fs_key}/mlp"
        )

        metrics = run_mlp(
            train_ds,
            val_ds,
            test_ds,
            n_feat,
            save_dir
        )

        fs_results["MLP"] = metrics

        print(
            f"    {fmt(metrics)}"
        )


        # ----------------------------------------------------
        # Store feature-set results
        # ----------------------------------------------------

        all_results[fs_key] = {

            "label": fs_label,

            "num_features": n_feat,

            "train_size": info["train_size"],
            "val_size": info["val_size"],
            "test_size": info["test_size"],

            "models": fs_results,
        }


    # ========================================================
    # AGGREGATED RESULTS
    # ========================================================

    print("\n\n")
    print("=" * 100)
    print(
        "  PHASE 2 RESULTS — "
        "3 MODELS × 2 FINBERT FEATURE SETS"
    )
    print("=" * 100)

    models = [
        "Logistic Regression",
        "LSTM",
        "MLP"
    ]

    header = (
        f"  {'Feature Set':<45s}"
    )

    for model in models:
        header += f"  {model:<24s}"

    print(header)

    print(
        f"  {'':45s}  "
        + ("Acc    F1   MCC   AUC     " * len(models))
    )

    print("-" * 100)


    for fs_key, res in all_results.items():

        row = (
            f"  {res['label']:<45s}"
        )

        for model in models:

            m = res["models"].get(
                model,
                {}
            )

            row += (
                f"  "
                f"{m.get('accuracy', 0):.3f} "
                f"{m.get('f1', 0):.3f} "
                f"{m.get('mcc', 0):+.3f} "
                f"{m.get('auc', 0.5):.3f}"
            )

        print(row)


    # ========================================================
    # SAVE SUMMARY
    # ========================================================

    result_dir = (
        "results/phase2_embeddings"
    )

    os.makedirs(
        result_dir,
        exist_ok=True
    )


    summary_flat = {}


    for fs_key, res in all_results.items():

        for model, metrics in res[
            "models"
        ].items():

            key = (
                f"{model} | "
                f"{res['label']}"
            )

            summary_flat[key] = {

                "accuracy":
                    metrics.get(
                        "accuracy", 0
                    ),

                "f1":
                    metrics.get(
                        "f1", 0
                    ),

                "mcc":
                    metrics.get(
                        "mcc", 0
                    ),

                "auc":
                    metrics.get(
                        "auc", 0.5
                    ),

                "n_samples":
                    metrics.get(
                        "n_samples", 0
                    ),

                "confusion_matrix":
                    metrics.get(
                        "confusion_matrix",
                        []
                    ),

                "feature_set":
                    res["label"],

                "model":
                    model,
            }


    with open(
        f"{result_dir}/summary.json",
        "w"
    ) as f:

        json.dump(
            summary_flat,
            f,
            indent=2
        )


    with open(
        f"{result_dir}/full_results.json",
        "w"
    ) as f:

        json.dump(
            all_results,
            f,
            indent=2
        )


    print("\n")
    print("=" * 80)
    print("Phase 2 experiments complete.")
    print(
        "Results saved to:",
        result_dir
    )
    print(
        "Total experiments:",
        len(models) * len(all_results)
    )
    print("=" * 80)


# ============================================================
# RUN
# ============================================================

run_phase2()



  Feature Set: Price + Company FinBERT
Split sizes:
  Train rows: 15969
  Val rows:   4359
  Test rows:  6275

  Features = 783
  Train = 15534 | Val = 3924 | Test = 5840

  [1/3] Logistic Regression
      LR input shape: train=(15534, 783), test=(5840, 783)
    Acc=0.4920  F1=0.5589  MCC=-0.0080  AUC=0.4953

  [2/3] LSTM (sliding window, W=5)
    Ep   1 | Train Acc=0.510 | Val Acc=0.484 MCC=0.0000
    Ep   5 | Train Acc=0.516 | Val Acc=0.479 MCC=-0.0274
    Ep  10 | Train Acc=0.521 | Val Acc=0.484 MCC=-0.0075
    Early stop at epoch 14
    Acc=0.5248  F1=0.4721  MCC=0.0459  AUC=0.5304

  [3/3] MLP
    Ep   1 | Train Acc=0.501 | Val Acc=0.481 MCC=-0.0215
    Ep   5 | Train Acc=0.525 | Val Acc=0.489 MCC=-0.0001
    Ep  10 | Train Acc=0.541 | Val Acc=0.490 MCC=-0.0009
    Ep  15 | Train Acc=0.561 | Val Acc=0.491 MCC=0.0008
    Ep  20 | Train Acc=0.583 | Val Acc=0.492 MCC=0.0013
    Early stop at epoch 24
    Acc=0.4830  F1=0.6109  MCC=-0.0221  AUC=0.5053


  Feature Set: Price + Fundam

## 13. Git Checkpoint

The Phase 2 result files are version-controlled.

Generated large data artifacts are intentionally excluded from Git:

- `company_embeddings.npy`
- `event_embeddings.npy`
- `dataset/stocknet_final_modeling_set_phase2.parquet`

The results directory contains the reproducible metric outputs for the six completed experiments.


In [50]:
!git status --short

 M .gitignore
?? results/phase2_embeddings/


In [49]:
%%writefile -a .gitignore

# Generated FinBERT embeddings
company_embeddings.npy
event_embeddings.npy

# Generated Phase 2 modeling dataset
dataset/stocknet_final_modeling_set_phase2.parquet

Appending to .gitignore


In [51]:
!git add results/phase2_embeddings/
!git status --short

 M .gitignore
A  results/phase2_embeddings/FS5_Price_CompanyEmbedding/logistic_regression/metrics.json
A  results/phase2_embeddings/FS5_Price_CompanyEmbedding/lstm/metrics.json
A  results/phase2_embeddings/FS5_Price_CompanyEmbedding/mlp/metrics.json
A  results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/logistic_regression/metrics.json
A  results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/lstm/metrics.json
A  results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/mlp/metrics.json
A  results/phase2_embeddings/full_results.json
A  results/phase2_embeddings/summary.json


In [54]:
!git commit -m "Add Phase 2 FinBERT embedding baselines"

[main e3e88da] Add Phase 2 FinBERT embedding baselines
 8 files changed, 364 insertions(+)
 create mode 100644 results/phase2_embeddings/FS5_Price_CompanyEmbedding/logistic_regression/metrics.json
 create mode 100644 results/phase2_embeddings/FS5_Price_CompanyEmbedding/lstm/metrics.json
 create mode 100644 results/phase2_embeddings/FS5_Price_CompanyEmbedding/mlp/metrics.json
 create mode 100644 results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/logistic_regression/metrics.json
 create mode 100644 results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/lstm/metrics.json
 create mode 100644 results/phase2_embeddings/FS6_Price_Fundamentals_CompanyEmbedding/mlp/metrics.json
 create mode 100644 results/phase2_embeddings/full_results.json
 create mode 100644 results/phase2_embeddings/summary.json


In [53]:
!git config --global user.email "aditya.m.hubli@gmail.com"
!git config --global user.name "Aditya-Hubli"

## 14. Current Results and Interpretation

### Phase 2 results

| Feature Set | Model | Accuracy | F1 | MCC | AUC |
|---|---|---:|---:|---:|---:|
| Price + Company FinBERT | Logistic Regression | 0.4920 | 0.5589 | -0.0080 | 0.4953 |
| Price + Company FinBERT | LSTM | **0.5248** | 0.4721 | **0.0459** | **0.5304** |
| Price + Company FinBERT | MLP | 0.4830 | **0.6109** | -0.0221 | 0.5053 |
| Price + Fundamentals + Company FinBERT | Logistic Regression | 0.4932 | 0.5606 | -0.0053 | 0.4961 |
| Price + Fundamentals + Company FinBERT | LSTM | 0.4870 | **0.6550** | 0.0000 | 0.5270 |
| Price + Fundamentals + Company FinBERT | MLP | 0.5038 | 0.5506 | 0.0140 | 0.5124 |

### Initial comparison

The strongest Phase 1 result was the **Price + Fundamentals LSTM** with approximately:

- MCC = 0.042
- AUC = 0.532

The strongest Phase 2 result so far is the **Price + Company FinBERT LSTM**:

- MCC = 0.0459
- AUC = 0.5304

The current experiment therefore does **not** show a substantial improvement from simply concatenating frozen 768-dimensional FinBERT company-text embeddings with the structured features.

This is a result about the **current representation strategy**, not a conclusion that financial text itself is uninformative.

Potential limitations to investigate next include the sparsity of text, mean pooling of multiple texts into one daily vector, and the large increase in input dimensionality.


In [55]:
!git log -1 --stat

commit e3e88da5d48e795fc0294575494b0004954fdb2c (HEAD -> main)
Author: Aditya-Hubli <aditya.m.hubli@gmail.com>
Date:   Mon Aug 24 06:56:20 2026 +0000

    Add Phase 2 FinBERT embedding baselines

 .../logistic_regression/metrics.json               |  19 ++++
 .../FS5_Price_CompanyEmbedding/lstm/metrics.json   |  21 ++++
 .../FS5_Price_CompanyEmbedding/mlp/metrics.json    |  21 ++++
 .../logistic_regression/metrics.json               |  19 ++++
 .../lstm/metrics.json                              |  21 ++++
 .../mlp/metrics.json                               |  21 ++++
 results/phase2_embeddings/full_results.json        | 126 +++++++++++++++++++++
 results/phase2_embeddings/summary.json             | 116 +++++++++++++++++++
 8 files changed, 364 insertions(+)


## 15. Next Steps

Before extending the architecture, validate the temporal semantics of `Company_Texts` and `Event_Texts` relative to the stock-day and prediction target.

Then investigate richer text representations and aggregation strategies, followed by the Event embedding experiments.

Planned directions include:

1. Temporal/data-alignment validation
2. Event embedding baselines
3. Alternative text aggregation strategies
4. Embedding dimensionality reduction
5. Multimodal fusion
6. Graph-based modeling
7. Federated learning

The current notebook should be treated as the **Phase 2 FinBERT company-text baseline checkpoint**.
